In [3]:
import pandas as pd
from PIL import Image
import os
from collections import defaultdict

def load_adversarial_dataset(csv_path, image_base_dir='/home/youssef-abuzeid/Uni/GP/org/Adversarial-Attacks-on-Deeplearning-Models/Attacks/CV/adversarial_images/'):
    """
    Reads the adversarial dataset from a CSV file and groups images by epsilon.

    Args:
        csv_path (str): The path to the 'adversarial_labels.csv' file.
        image_base_dir (str): The base directory where the image folders 
                              (e.g., 'eps_0.007/') are located.

    Returns:
        dict: A dictionary where keys are epsilon values (float) and values are
              lists of dictionaries, with each dictionary containing an 
              'image' (PIL.Image object), 'label' (int), and 'filename' (str).
    """
    print(f"Loading CSV from: {csv_path}")
    try:
        # 1. Read the CSV file into a pandas DataFrame
        df = pd.read_csv(csv_path)
    except FileNotFoundError:
        print(f"Error: The file {csv_path} was not found.")
        return None

    # 2. Group the DataFrame by the 'epsilon' column
    # This creates a collection of smaller DataFrames, one for each unique epsilon
    grouped_by_epsilon = df.groupby('epsilon')
    
    # Use defaultdict for convenience, so we don't have to check if a key exists
    adversarial_data = defaultdict(list)

    print("Processing images grouped by epsilon...")
    # 3. Iterate over each group (each unique epsilon value)
    for epsilon, group_df in grouped_by_epsilon:
        print(f"  -> Processing {len(group_df)} images for epsilon = {epsilon}")
        
        # 4. For each row in the current group, load the image
        for index, row in group_df.iterrows():
            filename = row['filename']
            label = row['label']
            
            # Construct the full path to the image file
            # os.path.join is used for cross-platform compatibility
            image_path = os.path.join(image_base_dir, filename)
            
            try:
                # Load the image using Pillow
                with Image.open(image_path) as img:
                    # We use img.copy() to load the image data into memory
                    # so the file can be closed by the 'with' statement.
                    loaded_image = img.copy()
                
                # 5. Append the loaded data to the list for the current epsilon
                adversarial_data[epsilon].append({
                    'image': loaded_image,
                    'label': int(label),
                    'filename': filename
                })

            except FileNotFoundError:
                print(f"    [Warning] Image not found, skipping: {image_path}")
            except Exception as e:
                print(f"    [Error] Could not load image {image_path}: {e}")

    # Convert defaultdict back to a regular dict for the final output
    return dict(adversarial_data)

# --- Main execution block ---
if __name__ == "__main__":
    CSV_FILE = '/home/youssef-abuzeid/Uni/GP/org/Adversarial-Attacks-on-Deeplearning-Models/Attacks/CV/adversarial_images/adversarial_labels.csv'
    
    # This function call does all the work
    adversarial_dataset = load_adversarial_dataset(CSV_FILE)
    
    if adversarial_dataset:
        print("\n--- Dataset Loading Complete ---")
        
        # Let's inspect the result
        print(f"Found data for {len(adversarial_dataset)} epsilon values: {list(adversarial_dataset.keys())}")
        

Loading CSV from: /home/youssef-abuzeid/Uni/GP/org/Adversarial-Attacks-on-Deeplearning-Models/Attacks/CV/adversarial_images/adversarial_labels.csv
Processing images grouped by epsilon...
  -> Processing 516 images for epsilon = 0.007
  -> Processing 516 images for epsilon = 0.01
  -> Processing 516 images for epsilon = 0.03
  -> Processing 516 images for epsilon = 0.1

--- Dataset Loading Complete ---
Found data for 4 epsilon values: [0.007, 0.01, 0.03, 0.1]


In [ ]:
print("Example data for epsilon = 0.007:")
if 0.007 in adversarial_dataset:
        example_data = adversarial_dataset[0.007][:5]  # Show first 5 examples
        for item in example_data:
            print(f"Filename: {item['filename']}, Label: {item['label']}")
else:
            print("No data found for epsilon = 0.007.")
            print("No data loaded.")

Example data for epsilon = 0.007:
Filename: <PIL.Image.Image image mode=RGB size=240x240 at 0x7FC118D827A0>, Label: 20
Filename: <PIL.Image.Image image mode=RGB size=240x240 at 0x7FC118D82530>, Label: 17
Filename: <PIL.Image.Image image mode=RGB size=240x240 at 0x7FC118D5F850>, Label: 29
Filename: <PIL.Image.Image image mode=RGB size=240x240 at 0x7FC118D5F820>, Label: 4
Filename: <PIL.Image.Image image mode=RGB size=240x240 at 0x7FC118D5FBB0>, Label: 0


In [8]:
import cv2
import numpy as np

def jpeg_compress_opencv(numpy_image, quality=85):
    """
    Applies JPEG compression to a NumPy image array entirely in memory.

    Args:
        numpy_image (np.array): Input image in OpenCV format (BGR, uint8, 0-255).
        quality (int): JPEG quality (0-100).

    Returns:
        np.array: The image after compression/decompression.
    """
    # 1. Define encoding parameters for JPEG quality.
    encode_param = [int(cv2.IMWRITE_JPEG_QUALITY), quality]

    # 2. Encode the image into a JPEG byte stream in memory.
    result, encoded_image = cv2.imencode('.jpg', numpy_image, encode_param)

    if not result:
        raise ValueError("Could not encode image to JPEG")

    # 3. Decode the byte stream back into a NumPy image array.
    decoded_image = cv2.imdecode(encoded_image, 1) # 1 means load as color image

    return decoded_image

# --- Example Usage ---
# Assume `original_image_cv` is your input image loaded with OpenCV
# original_image_cv = cv2.imread("path/to/your/image.png")

# Apply the defense
# defensive_image_cv = jpeg_compress_opencv(original_image_cv)

# ... then preprocess and feed to the model

In [9]:
def apply_jpeg_defense_to_dataset(dataset, quality=85):
    """
    Applies the JPEG defense to every image in the loaded dataset.

    Args:
        dataset (dict): The dataset loaded by `load_adversarial_dataset`.
        quality (int): The JPEG quality setting for the defense.

    Returns:
        dict: A new dataset with the same structure, but with defended images.
    """
    defended_dataset = defaultdict(list)
    print(f"\nApplying JPEG compression (quality={quality}) to all images...")

    # Iterate through each epsilon group
    for epsilon, image_data_list in dataset.items():
        print(f"  -> Processing {len(image_data_list)} images for epsilon = {epsilon}")
        
        # Iterate through each image in the group
        for data_item in image_data_list:
            original_pil_image = data_item['image']
            
            # 1. Convert PIL Image (RGB) to NumPy array
            numpy_rgb = np.array(original_pil_image)
            
            # 2. Convert RGB color space to BGR for OpenCV
            numpy_bgr = cv2.cvtColor(numpy_rgb, cv2.COLOR_RGB2BGR)
            
            # 3. Apply the OpenCV-based JPEG compression defense
            defended_numpy_bgr = jpeg_compress_opencv(numpy_bgr, quality=quality)
            
            # 4. Convert the BGR result back to RGB
            defended_numpy_rgb = cv2.cvtColor(defended_numpy_bgr, cv2.COLOR_BGR2RGB)
            
            # 5. Convert the defended NumPy array back to a PIL Image
            defended_pil_image = Image.fromarray(defended_numpy_rgb)
            
            # 6. Store the new, defended image in our results dictionary
            defended_dataset[epsilon].append({
                'image': defended_pil_image,
                'label': data_item['label'],
                'filename': data_item['filename']
            })
            
    return dict(defended_dataset)


In [10]:
defended_dataset = apply_jpeg_defense_to_dataset(adversarial_dataset, quality=75)


Applying JPEG compression (quality=75) to all images...
  -> Processing 516 images for epsilon = 0.007
  -> Processing 516 images for epsilon = 0.01
  -> Processing 516 images for epsilon = 0.03
  -> Processing 516 images for epsilon = 0.1


In [21]:
from tqdm import tqdm # For a nice progress bar
def load_and_preprocess_image(path, target_size, apply_defense=False, jpeg_quality=75):
    """
    Loads an image and prepares it for the EfficientNetB1 model.
    """
    path = "/home/youssef-abuzeid/Uni/GP/org/Adversarial-Attacks-on-Deeplearning-Models/Attacks/CV/adversarial_images/"+ path
    with Image.open(path) as img:
        pil_image = img.convert('RGB').copy()
    
    if apply_defense:
        numpy_rgb = np.array(pil_image)
        numpy_bgr = cv2.cvtColor(numpy_rgb, cv2.COLOR_RGB2BGR)
        defended_bgr = jpeg_compress_opencv(numpy_bgr, quality=jpeg_quality)
        defended_rgb = cv2.cvtColor(defended_bgr, cv2.COLOR_BGR2RGB)
        pil_image = Image.fromarray(defended_rgb)

    # Pre-processing for the model
    resized_image = pil_image.resize(target_size)
    img_array = np.array(resized_image)
    img_batch = np.expand_dims(img_array, axis=0)
    
    # ### IMPORTANT MODIFICATION ###
    # The model expects pixel values in the [0, 255] range.
    # We DO NOT use a `preprocess_input` function here because that would
    # normalize the values. We just ensure the data type is correct.
    return img_batch.astype(np.float32)

def evaluate_model(model, dataset_info, apply_defense, jpeg_quality=75):
    """Evaluates model accuracy on a dataset, with or without defense."""
    results = {}
    target_size = model.input_shape[1:3] # Get (height, width) from model

    for epsilon, items in dataset_info.items():
        correct_predictions = 0
        description = f"Epsilon {epsilon} ({'Defended' if apply_defense else 'Adversarial'})"
        for item in tqdm(items, desc=description):
            image_path = item['filename']
            image_batch = load_and_preprocess_image(
                image_path, 
                target_size, 
                apply_defense=apply_defense, 
                jpeg_quality=jpeg_quality
            )
            
            true_label = item['label']
            
            
            predictions = model.predict(image_batch, verbose=0)
            predicted_label = np.argmax(predictions[0])
            
            if predicted_label == true_label:
                correct_predictions += 1
        
        accuracy = correct_predictions / len(items)
        results[epsilon] = accuracy
        
    return results

In [23]:
from tensorflow.keras.models import load_model

model = load_model('../model.keras')

adversarial_dataset = load_adversarial_dataset('/home/youssef-abuzeid/Uni/GP/org/Adversarial-Attacks-on-Deeplearning-Models/Attacks/CV/adversarial_images_PGD/adversarial_labels.csv')

# Evaluate the model on the original adversarial dataset
original_results = evaluate_model(model, adversarial_dataset, apply_defense=False)

# Evaluate the model on the defended dataset
defended_results = evaluate_model(model, adversarial_dataset, apply_defense=True, jpeg_quality=75)
# Print the results
print("\n--- Evaluation Results ---")
print("Original Adversarial Dataset Accuracy:")
for epsilon, accuracy in original_results.items():
    print(f"Epsilon {epsilon}: {accuracy:.4f}")
print("\nDefended Dataset Accuracy (JPEG Compression):")
for epsilon, accuracy in defended_results.items():
    print(f"Epsilon {epsilon}: {accuracy:.4f}")
print("\n--- Evaluation Complete ---")



Loading CSV from: /home/youssef-abuzeid/Uni/GP/org/Adversarial-Attacks-on-Deeplearning-Models/Attacks/CV/adversarial_images_PGD/adversarial_labels.csv
Processing images grouped by epsilon...
  -> Processing 516 images for epsilon = 0.007
  -> Processing 516 images for epsilon = 0.01
  -> Processing 516 images for epsilon = 0.03
  -> Processing 516 images for epsilon = 0.1


Epsilon 0.007 (Adversarial):   0%|          | 0/516 [00:00<?, ?it/s]/home/youssef-abuzeid/.local/lib/python3.10/site-packages/keras/src/models/functional.py:238: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: ['Input_Layer']
Received: inputs=Tensor(shape=(1, 240, 240, 3))
  warnings.warn(msg)
Epsilon 0.1 (Defended): 100%|██████████| 516/516 [02:04<00:00,  4.16it/s]


--- Evaluation Results ---
Original Adversarial Dataset Accuracy:
Epsilon 0.007: 0.0252
Epsilon 0.01: 0.0252
Epsilon 0.03: 0.0194
Epsilon 0.1: 0.0271

Defended Dataset Accuracy (JPEG Compression):
Epsilon 0.007: 0.0252
Epsilon 0.01: 0.0252
Epsilon 0.03: 0.0213
Epsilon 0.1: 0.0271

--- Evaluation Complete ---


In [1]:
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from tensorflow.keras.models import load_model

loaded_model_path = '../model.keras'
model = load_model(loaded_model_path)
# Load the CSV file
csv_path = '/home/youssef-abuzeid/Uni/GP/org/Adversarial-Attacks-on-Deeplearning-Models/Attacks/CV/Attacks Evaluation on Testset/adversarial_images_FGSM_new/adversarial_labels.csv'
df = pd.read_csv(csv_path)

# List of epsilon values to evaluate
epsilons = [0.007, 0.01, 0.03, 0.1]
for eps in epsilons:
    print(f"\n{'='*50}\nEvaluating Adversarial Images for Epsilon = {eps}\n{'='*50}")

    # Filter CSV for the current epsilon
    eps_df = df[df['epsilon'] == eps].copy()

    # Remove the 'eps_X.XXX/' prefix from filenames (if present)
    eps_df['filename'] = eps_df['filename'].str.replace(f'eps_{eps}/', '')

    # Create ImageDataGenerator (no normalization)
    test_datagen = ImageDataGenerator()

    # Prepare test generator
    test_generator = test_datagen.flow_from_dataframe(
        dataframe=eps_df,
        directory=f'/home/youssef-abuzeid/Uni/GP/org/Adversarial-Attacks-on-Deeplearning-Models/Attacks/CV/Attacks Evaluation on Testset/adversarial_images_FGSM_new/eps_{eps}',  # Folder for current epsilon
        x_col='filename',
        y_col='label',
        target_size=(240, 240),  # Adjust to your model's input size
        batch_size=16,
        class_mode='raw',  # For integer labels
        shuffle=False  # Keep order aligned with CSV
    )

    # Skip if no images found
    if test_generator.samples == 0:
        print(f" No images found for epsilon={eps}. Check paths or filenames.")
        continue

    # Get model predictions
    predictions = model.predict(test_generator)
    y_pred = tf.argmax(predictions, axis=1).numpy()
    y_true = test_generator.labels

    # Calculate metrics
    acc = accuracy_score(y_true, y_pred)
    print(f"\nPost-Adversarial Accuracy (ε={eps}): {acc:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred))
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_true, y_pred))

2025-06-19 11:44:14.175076: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1750322654.241997   30438 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1750322654.263242   30438 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1750322654.401694   30438 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1750322654.401718   30438 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1750322654.401719   30438 computation_placer.cc:177] computation placer alr


Evaluating Adversarial Images for Epsilon = 0.007
Found 516 validated image filenames.


/home/youssef-abuzeid/.local/lib/python3.10/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()
/home/youssef-abuzeid/.local/lib/python3.10/site-packages/keras/src/models/functional.py:238: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: ['Input_Layer']
Received: inputs=Tensor(shape=(16, 240, 240, 3))
  warnings.warn(msg)
I0000 00:00:1750322664.853463   32684 service.cc:152] XLA service 0x705d34003e10 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1750322664.862518   32684 service.cc:160]   StreamExecutor device (0): NVIDIA GeForce GTX 1660 Ti, Compute Capability 7.5
2025-06-19 11:44:25.022666: I tensorflow

 5/33 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step

I0000 00:00:1750322671.382175   32684 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


...

/home/youssef-abuzeid/.local/lib/python3.10/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


33/33 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step

Post-Adversarial Accuracy (ε=0.03): 0.1279

Classification Report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00        12
           1       0.15      0.17      0.16        12
           2       0.10      0.17      0.12        12
           3       0.00      0.00      0.00        12
           4       0.00      0.00      0.00        12
           5       0.00      0.00      0.00        12
           6       0.00      0.00      0.00        12
           7       0.00      0.00      0.00        12
           8       0.00      0.00      0.00        12
           9       0.05      0.08      0.06        12
          10       0.00      0.00      0.00        12
          11       0.00      0.00      0.00        12
          12       0.18      0.17      0.17        12
          13       0.71      0.83      0.77        12
          14       0.25      0.17      0.20        12
          15       0.20    

/home/youssef-abuzeid/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/youssef-abuzeid/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/youssef-abuzeid/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(resu

33/33 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step

Post-Adversarial Accuracy (ε=0.1): 0.0930

Classification Report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00        12
           1       0.00      0.00      0.00        12
           2       0.07      0.33      0.12        12
           3       0.00      0.00      0.00        12
           4       0.00      0.00      0.00        12
           5       0.00      0.00      0.00        12
           6       0.00      0.00      0.00        12
           7       0.00      0.00      0.00        12
           8       0.00      0.00      0.00        12
           9       0.00      0.00      0.00        12
          10       0.00      0.00      0.00        12
          11       0.00      0.00      0.00        12
          12       0.14      0.17      0.15        12
          13       0.50      0.50      0.50        12
          14       0.06      0.17      0.09        12
          15       0.00     

/home/youssef-abuzeid/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/youssef-abuzeid/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/youssef-abuzeid/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(resu

In [5]:
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import accuracy_score
from tensorflow.keras.models import load_model
import numpy as np
import cv2
import os

# --- Configuration ---
JPEG_QUALITY = 75
TARGET_SIZE = (240, 240)
BATCH_SIZE = 16

# --- Helper Functions for the Defense ---

def jpeg_compress_opencv(numpy_image_bgr, quality=75):
    """Applies in-memory JPEG compression to a BGR NumPy image array."""
    encode_param = [int(cv2.IMWRITE_JPEG_QUALITY), quality]
    result, encoded_image = cv2.imencode('.jpg', numpy_image_bgr, encode_param)
    if not result:
        raise ValueError("Could not encode image to JPEG")
    return cv2.imdecode(encoded_image, 1)

def jpeg_defense_preprocessor(image_rgb):
    """
    A preprocessing function for ImageDataGenerator that applies the JPEG defense.
    It handles the required RGB <-> BGR color space conversions.
    
    Args:
        image_rgb (np.array): An image in RGB format (from ImageDataGenerator).
    
    Returns:
        np.array: The defended image, also in RGB format.
    """
    # 1. Convert from RGB (Keras default) to BGR (OpenCV default)
    image_bgr = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR)
    
    # 2. Apply the JPEG compression defense on the BGR image
    defended_bgr = jpeg_compress_opencv(image_bgr, quality=JPEG_QUALITY)
    
    # 3. Convert back from BGR to RGB before returning to Keras
    defended_rgb = cv2.cvtColor(defended_bgr, cv2.COLOR_BGR2RGB)
    
    return defended_rgb

# --- Main Evaluation Script ---

# Load the model
# loaded_model_path = '../model.keras'
loaded_model_path = '/home/youssef-abuzeid/Uni/GP/org/Adversarial-Attacks-on-Deeplearning-Models/Attacks/CV/adversarial_images_PGD/../model.keras'
model = load_model(loaded_model_path)

# Load the CSV file
csv_path = '/home/youssef-abuzeid/Uni/GP/org/Adversarial-Attacks-on-Deeplearning-Models/Attacks/CV/Attacks Evaluation on Testset/adversarial_images_FGSM_new/adversarial_labels.csv'
df = pd.read_csv(csv_path)

# --- Evaluation Loop ---
results_summary = []
epsilons = [0.007, 0.01, 0.03, 0.1]

for eps in epsilons:
    print(f"\n{'='*60}\nEvaluating for Epsilon = {eps}\n{'='*60}")

    # Filter CSV for the current epsilon
    eps_df = df[df['epsilon'] == eps].copy()
    
    # Define paths
    base_dir = '/home/youssef-abuzeid/Uni/GP/org/Adversarial-Attacks-on-Deeplearning-Models/Attacks/CV/Attacks Evaluation on Testset/adversarial_images_FGSM_new'
    image_directory = os.path.join(base_dir, f'eps_{eps}')
    
    # The user's code cleans filenames this way, which is fine, but we need
    # the full path for the generator. The `directory` parameter handles this.
    eps_df['filename'] = eps_df['filename'].str.replace(f'eps_{eps}/', '')

    if eps_df.empty:
        print(f"No data found for epsilon={eps}. Skipping.")
        continue

    # --- 1. Evaluate WITHOUT defense ---
    print("\n--- Evaluating Adversarial Images (No Defense) ---")
    adversarial_datagen = ImageDataGenerator() # No preprocessing function
    
    adversarial_generator = adversarial_datagen.flow_from_dataframe(
        dataframe=eps_df,
        directory=image_directory,
        x_col='filename',
        y_col='label',
        target_size=TARGET_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='raw',
        shuffle=False
    )
    
    # Get predictions and true labels
    adv_predictions = model.predict(adversarial_generator)
    y_pred_adv = tf.argmax(adv_predictions, axis=1).numpy()
    y_true = adversarial_generator.labels
    
    # Calculate and store accuracy
    adv_acc = accuracy_score(y_true, y_pred_adv)
    print(f"Accuracy on Adversarial Images: {adv_acc:.4f}")

    # --- 2. Evaluate WITH JPEG defense ---
    print(f"\n--- Evaluating with JPEG Defense (Quality={JPEG_QUALITY}) ---")
    defended_datagen = ImageDataGenerator(
        preprocessing_function=jpeg_defense_preprocessor
    )
    
    defended_generator = defended_datagen.flow_from_dataframe(
        dataframe=eps_df,
        directory=image_directory,
        x_col='filename',
        y_col='label',
        target_size=TARGET_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='raw',
        shuffle=False
    )
    
    # Get predictions and true labels
    def_predictions = model.predict(defended_generator)
    y_pred_def = tf.argmax(def_predictions, axis=1).numpy()
    y_true_def = defended_generator.labels # Should be identical to y_true
    
    # Calculate and store accuracy
    def_acc = accuracy_score(y_true_def, y_pred_def)
    print(f"Accuracy on Defended Images: {def_acc:.4f}")
    
    # Store results for the final report
    results_summary.append({
        "Epsilon": eps,
        "Accuracy (Adversarial)": adv_acc,
        "Accuracy (Defended)": def_acc
    })

# --- Final Report ---
print(f"\n\n{'='*65}\nFINAL ACCURACY REPORT (JPEG DEFENSE, Q={JPEG_QUALITY})\n{'='*65}")
report_df = pd.DataFrame(results_summary)
report_df['Improvement'] = report_df['Accuracy (Defended)'] - report_df['Accuracy (Adversarial)']

# Formatting for clear presentation
report_df['Accuracy (Adversarial)'] = report_df['Accuracy (Adversarial)'].apply(lambda x: f"{x:.2%}")
report_df['Accuracy (Defended)'] = report_df['Accuracy (Defended)'].apply(lambda x: f"{x:.2%}")
report_df['Improvement'] = report_df['Improvement'].apply(lambda x: f"+{x:.2%}" if x >= 0 else f"{x:.2%}")

print(report_df.to_string(index=False))
print("="*65)


Evaluating for Epsilon = 0.007

--- Evaluating Adversarial Images (No Defense) ---
Found 516 validated image filenames.


/home/youssef-abuzeid/.local/lib/python3.10/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()
/home/youssef-abuzeid/.local/lib/python3.10/site-packages/keras/src/models/functional.py:238: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: ['Input_Layer']
Received: inputs=Tensor(shape=(16, 240, 240, 3))
  warnings.warn(msg)


31/33 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step

/home/youssef-abuzeid/.local/lib/python3.10/site-packages/keras/src/models/functional.py:238: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: ['Input_Layer']
Received: inputs=Tensor(shape=(None, 240, 240, 3))
  warnings.warn(msg)


33/33 ━━━━━━━━━━━━━━━━━━━━ 12s 202ms/step
Accuracy on Adversarial Images: 0.4496

--- Evaluating with JPEG Defense (Quality=75) ---
Found 516 validated image filenames.
 1/33 ━━━━━━━━━━━━━━━━━━━━ 2s 76ms/step

...

33/33 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step
Accuracy on Defended Images: 0.2984

Evaluating for Epsilon = 0.1

--- Evaluating Adversarial Images (No Defense) ---
Found 516 validated image filenames.
 3/33 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step

/home/youssef-abuzeid/.local/lib/python3.10/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


33/33 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step
Accuracy on Adversarial Images: 0.0930

--- Evaluating with JPEG Defense (Quality=75) ---
Found 516 validated image filenames.
 1/33 ━━━━━━━━━━━━━━━━━━━━ 2s 70ms/step

/home/youssef-abuzeid/.local/lib/python3.10/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


33/33 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step
Accuracy on Defended Images: 0.1027


FINAL ACCURACY REPORT (JPEG DEFENSE, Q=75)
 Epsilon Accuracy (Adversarial) Accuracy (Defended) Improvement
   0.007                 44.96%              78.88%     +33.91%
   0.010                 34.30%              68.80%     +34.50%
   0.030                 12.79%              29.84%     +17.05%
   0.100                  9.30%              10.27%      +0.97%


In [6]:
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import accuracy_score
from tensorflow.keras.models import load_model
import numpy as np
import cv2
import os

# --- Configuration ---
JPEG_QUALITY = 50
TARGET_SIZE = (240, 240)
BATCH_SIZE = 16

# --- Helper Functions for the Defense ---

def jpeg_compress_opencv(numpy_image_bgr, quality=75):
    """Applies in-memory JPEG compression to a BGR NumPy image array."""
    encode_param = [int(cv2.IMWRITE_JPEG_QUALITY), quality]
    result, encoded_image = cv2.imencode('.jpg', numpy_image_bgr, encode_param)
    if not result:
        raise ValueError("Could not encode image to JPEG")
    return cv2.imdecode(encoded_image, 1)

def jpeg_defense_preprocessor(image_rgb):
    """
    A preprocessing function for ImageDataGenerator that applies the JPEG defense.
    It handles the required RGB <-> BGR color space conversions.
    
    Args:
        image_rgb (np.array): An image in RGB format (from ImageDataGenerator).
    
    Returns:
        np.array: The defended image, also in RGB format.
    """
    # 1. Convert from RGB (Keras default) to BGR (OpenCV default)
    image_bgr = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR)
    
    # 2. Apply the JPEG compression defense on the BGR image
    defended_bgr = jpeg_compress_opencv(image_bgr, quality=JPEG_QUALITY)
    
    # 3. Convert back from BGR to RGB before returning to Keras
    defended_rgb = cv2.cvtColor(defended_bgr, cv2.COLOR_BGR2RGB)
    
    return defended_rgb

# --- Main Evaluation Script ---

# Load the model
# loaded_model_path = '../model.keras'
loaded_model_path = '/home/youssef-abuzeid/Uni/GP/org/Adversarial-Attacks-on-Deeplearning-Models/Attacks/CV/adversarial_images_PGD/../model.keras'
model = load_model(loaded_model_path)

# Load the CSV file
csv_path = '/home/youssef-abuzeid/Uni/GP/org/Adversarial-Attacks-on-Deeplearning-Models/Attacks/CV/Attacks Evaluation on Testset/adversarial_images_FGSM_new/adversarial_labels.csv'
df = pd.read_csv(csv_path)

# --- Evaluation Loop ---
results_summary = []
epsilons = [0.007, 0.01, 0.03, 0.1]

for eps in epsilons:
    print(f"\n{'='*60}\nEvaluating for Epsilon = {eps}\n{'='*60}")

    # Filter CSV for the current epsilon
    eps_df = df[df['epsilon'] == eps].copy()
    
    # Define paths
    base_dir = '/home/youssef-abuzeid/Uni/GP/org/Adversarial-Attacks-on-Deeplearning-Models/Attacks/CV/Attacks Evaluation on Testset/adversarial_images_FGSM_new'
    image_directory = os.path.join(base_dir, f'eps_{eps}')
    
    # The user's code cleans filenames this way, which is fine, but we need
    # the full path for the generator. The `directory` parameter handles this.
    eps_df['filename'] = eps_df['filename'].str.replace(f'eps_{eps}/', '')

    if eps_df.empty:
        print(f"No data found for epsilon={eps}. Skipping.")
        continue

    # --- 1. Evaluate WITHOUT defense ---
    print("\n--- Evaluating Adversarial Images (No Defense) ---")
    adversarial_datagen = ImageDataGenerator() # No preprocessing function
    
    adversarial_generator = adversarial_datagen.flow_from_dataframe(
        dataframe=eps_df,
        directory=image_directory,
        x_col='filename',
        y_col='label',
        target_size=TARGET_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='raw',
        shuffle=False
    )
    
    # Get predictions and true labels
    adv_predictions = model.predict(adversarial_generator)
    y_pred_adv = tf.argmax(adv_predictions, axis=1).numpy()
    y_true = adversarial_generator.labels
    
    # Calculate and store accuracy
    adv_acc = accuracy_score(y_true, y_pred_adv)
    print(f"Accuracy on Adversarial Images: {adv_acc:.4f}")

    # --- 2. Evaluate WITH JPEG defense ---
    print(f"\n--- Evaluating with JPEG Defense (Quality={JPEG_QUALITY}) ---")
    defended_datagen = ImageDataGenerator(
        preprocessing_function=jpeg_defense_preprocessor
    )
    
    defended_generator = defended_datagen.flow_from_dataframe(
        dataframe=eps_df,
        directory=image_directory,
        x_col='filename',
        y_col='label',
        target_size=TARGET_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='raw',
        shuffle=False
    )
    
    # Get predictions and true labels
    def_predictions = model.predict(defended_generator)
    y_pred_def = tf.argmax(def_predictions, axis=1).numpy()
    y_true_def = defended_generator.labels # Should be identical to y_true
    
    # Calculate and store accuracy
    def_acc = accuracy_score(y_true_def, y_pred_def)
    print(f"Accuracy on Defended Images: {def_acc:.4f}")
    
    # Store results for the final report
    results_summary.append({
        "Epsilon": eps,
        "Accuracy (Adversarial)": adv_acc,
        "Accuracy (Defended)": def_acc
    })

# --- Final Report ---
print(f"\n\n{'='*65}\nFINAL ACCURACY REPORT (JPEG DEFENSE, Q={JPEG_QUALITY})\n{'='*65}")
report_df = pd.DataFrame(results_summary)
report_df['Improvement'] = report_df['Accuracy (Defended)'] - report_df['Accuracy (Adversarial)']

# Formatting for clear presentation
report_df['Accuracy (Adversarial)'] = report_df['Accuracy (Adversarial)'].apply(lambda x: f"{x:.2%}")
report_df['Accuracy (Defended)'] = report_df['Accuracy (Defended)'].apply(lambda x: f"{x:.2%}")
report_df['Improvement'] = report_df['Improvement'].apply(lambda x: f"+{x:.2%}" if x >= 0 else f"{x:.2%}")

print(report_df.to_string(index=False))
print("="*65)


Evaluating for Epsilon = 0.007

--- Evaluating Adversarial Images (No Defense) ---
Found 516 validated image filenames.


/home/youssef-abuzeid/.local/lib/python3.10/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()
/home/youssef-abuzeid/.local/lib/python3.10/site-packages/keras/src/models/functional.py:238: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: ['Input_Layer']
Received: inputs=Tensor(shape=(16, 240, 240, 3))
  warnings.warn(msg)


31/33 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step

/home/youssef-abuzeid/.local/lib/python3.10/site-packages/keras/src/models/functional.py:238: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: ['Input_Layer']
Received: inputs=Tensor(shape=(None, 240, 240, 3))
  warnings.warn(msg)


33/33 ━━━━━━━━━━━━━━━━━━━━ 12s 204ms/step
Accuracy on Adversarial Images: 0.4496

--- Evaluating with JPEG Defense (Quality=50) ---
Found 516 validated image filenames.
 1/33 ━━━━━━━━━━━━━━━━━━━━ 2s 66ms/step

...

33/33 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step
Accuracy on Defended Images: 0.3508

Evaluating for Epsilon = 0.1

--- Evaluating Adversarial Images (No Defense) ---
Found 516 validated image filenames.
 4/33 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step

/home/youssef-abuzeid/.local/lib/python3.10/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


33/33 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step
Accuracy on Adversarial Images: 0.0930

--- Evaluating with JPEG Defense (Quality=50) ---
Found 516 validated image filenames.
 4/33 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step

/home/youssef-abuzeid/.local/lib/python3.10/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


33/33 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step
Accuracy on Defended Images: 0.1066


FINAL ACCURACY REPORT (JPEG DEFENSE, Q=50)
 Epsilon Accuracy (Adversarial) Accuracy (Defended) Improvement
   0.007                 44.96%              80.04%     +35.08%
   0.010                 34.30%              69.96%     +35.66%
   0.030                 12.79%              35.08%     +22.29%
   0.100                  9.30%              10.66%      +1.36%


In [7]:
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import accuracy_score
from tensorflow.keras.models import load_model
import numpy as np
import cv2
import os

# --- Configuration ---
JPEG_QUALITY = 75
TARGET_SIZE = (240, 240)
BATCH_SIZE = 16

# --- Helper Functions for the Defense ---

def jpeg_compress_opencv(numpy_image_bgr, quality=75):
    """Applies in-memory JPEG compression to a BGR NumPy image array."""
    encode_param = [int(cv2.IMWRITE_JPEG_QUALITY), quality]
    result, encoded_image = cv2.imencode('.jpg', numpy_image_bgr, encode_param)
    if not result:
        raise ValueError("Could not encode image to JPEG")
    return cv2.imdecode(encoded_image, 1)

def jpeg_defense_preprocessor(image_rgb):
    """
    A preprocessing function for ImageDataGenerator that applies the JPEG defense.
    It handles the required RGB <-> BGR color space conversions.
    
    Args:
        image_rgb (np.array): An image in RGB format (from ImageDataGenerator).
    
    Returns:
        np.array: The defended image, also in RGB format.
    """
    # 1. Convert from RGB (Keras default) to BGR (OpenCV default)
    image_bgr = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR)
    
    # 2. Apply the JPEG compression defense on the BGR image
    defended_bgr = jpeg_compress_opencv(image_bgr, quality=JPEG_QUALITY)
    
    # 3. Convert back from BGR to RGB before returning to Keras
    defended_rgb = cv2.cvtColor(defended_bgr, cv2.COLOR_BGR2RGB)
    
    return defended_rgb

# --- Main Evaluation Script ---

# Load the model
# loaded_model_path = '../model.keras'
loaded_model_path = '/home/youssef-abuzeid/Uni/GP/org/Adversarial-Attacks-on-Deeplearning-Models/Attacks/CV/adversarial_images_PGD/../model.keras'
model = load_model(loaded_model_path)

# Load the CSV file
csv_path = '/home/youssef-abuzeid/Uni/GP/org/Adversarial-Attacks-on-Deeplearning-Models/Attacks/CV/Attacks Evaluation on Testset/adversarial_images_PGD_new/adversarial_labels.csv'
df = pd.read_csv(csv_path)

# --- Evaluation Loop ---
results_summary = []
epsilons = [0.007, 0.01, 0.03, 0.1]

for eps in epsilons:
    print(f"\n{'='*60}\nEvaluating for Epsilon = {eps}\n{'='*60}")

    # Filter CSV for the current epsilon
    eps_df = df[df['epsilon'] == eps].copy()
    
    # Define paths
    base_dir = '/home/youssef-abuzeid/Uni/GP/org/Adversarial-Attacks-on-Deeplearning-Models/Attacks/CV/Attacks Evaluation on Testset/adversarial_images_PGD_new'
    image_directory = os.path.join(base_dir, f'eps_{eps}')
    
    # The user's code cleans filenames this way, which is fine, but we need
    # the full path for the generator. The `directory` parameter handles this.
    eps_df['filename'] = eps_df['filename'].str.replace(f'eps_{eps}/', '')

    if eps_df.empty:
        print(f"No data found for epsilon={eps}. Skipping.")
        continue

    # --- 1. Evaluate WITHOUT defense ---
    print("\n--- Evaluating Adversarial Images (No Defense) ---")
    adversarial_datagen = ImageDataGenerator() # No preprocessing function
    
    adversarial_generator = adversarial_datagen.flow_from_dataframe(
        dataframe=eps_df,
        directory=image_directory,
        x_col='filename',
        y_col='label',
        target_size=TARGET_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='raw',
        shuffle=False
    )
    
    # Get predictions and true labels
    adv_predictions = model.predict(adversarial_generator)
    y_pred_adv = tf.argmax(adv_predictions, axis=1).numpy()
    y_true = adversarial_generator.labels
    
    # Calculate and store accuracy
    adv_acc = accuracy_score(y_true, y_pred_adv)
    print(f"Accuracy on Adversarial Images: {adv_acc:.4f}")

    # --- 2. Evaluate WITH JPEG defense ---
    print(f"\n--- Evaluating with JPEG Defense (Quality={JPEG_QUALITY}) ---")
    defended_datagen = ImageDataGenerator(
        preprocessing_function=jpeg_defense_preprocessor
    )
    
    defended_generator = defended_datagen.flow_from_dataframe(
        dataframe=eps_df,
        directory=image_directory,
        x_col='filename',
        y_col='label',
        target_size=TARGET_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='raw',
        shuffle=False
    )
    
    # Get predictions and true labels
    def_predictions = model.predict(defended_generator)
    y_pred_def = tf.argmax(def_predictions, axis=1).numpy()
    y_true_def = defended_generator.labels # Should be identical to y_true
    
    # Calculate and store accuracy
    def_acc = accuracy_score(y_true_def, y_pred_def)
    print(f"Accuracy on Defended Images: {def_acc:.4f}")
    
    # Store results for the final report
    results_summary.append({
        "Epsilon": eps,
        "Accuracy (Adversarial)": adv_acc,
        "Accuracy (Defended)": def_acc
    })

# --- Final Report ---
print(f"\n\n{'='*65}\nFINAL ACCURACY REPORT (JPEG DEFENSE, Q={JPEG_QUALITY})\n{'='*65}")
report_df = pd.DataFrame(results_summary)
report_df['Improvement'] = report_df['Accuracy (Defended)'] - report_df['Accuracy (Adversarial)']

# Formatting for clear presentation
report_df['Accuracy (Adversarial)'] = report_df['Accuracy (Adversarial)'].apply(lambda x: f"{x:.2%}")
report_df['Accuracy (Defended)'] = report_df['Accuracy (Defended)'].apply(lambda x: f"{x:.2%}")
report_df['Improvement'] = report_df['Improvement'].apply(lambda x: f"+{x:.2%}" if x >= 0 else f"{x:.2%}")

print(report_df.to_string(index=False))
print("="*65)


Evaluating for Epsilon = 0.007

--- Evaluating Adversarial Images (No Defense) ---
Found 516 validated image filenames.


/home/youssef-abuzeid/.local/lib/python3.10/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()
/home/youssef-abuzeid/.local/lib/python3.10/site-packages/keras/src/models/functional.py:238: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: ['Input_Layer']
Received: inputs=Tensor(shape=(16, 240, 240, 3))
  warnings.warn(msg)


31/33 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step

/home/youssef-abuzeid/.local/lib/python3.10/site-packages/keras/src/models/functional.py:238: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: ['Input_Layer']
Received: inputs=Tensor(shape=(None, 240, 240, 3))
  warnings.warn(msg)


33/33 ━━━━━━━━━━━━━━━━━━━━ 15s 254ms/step
Accuracy on Adversarial Images: 0.3643

--- Evaluating with JPEG Defense (Quality=75) ---
Found 516 validated image filenames.


...

33/33 ━━━━━━━━━━━━━━━━━━━━ 2s 49ms/step
Accuracy on Defended Images: 0.2539

Evaluating for Epsilon = 0.1

--- Evaluating Adversarial Images (No Defense) ---
Found 516 validated image filenames.


/home/youssef-abuzeid/.local/lib/python3.10/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


33/33 ━━━━━━━━━━━━━━━━━━━━ 1s 39ms/step
Accuracy on Adversarial Images: 0.0000

--- Evaluating with JPEG Defense (Quality=75) ---
Found 516 validated image filenames.


/home/youssef-abuzeid/.local/lib/python3.10/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


33/33 ━━━━━━━━━━━━━━━━━━━━ 2s 52ms/step
Accuracy on Defended Images: 0.0349


FINAL ACCURACY REPORT (JPEG DEFENSE, Q=75)
 Epsilon Accuracy (Adversarial) Accuracy (Defended) Improvement
   0.007                 36.43%              73.26%     +36.82%
   0.010                 21.71%              62.21%     +40.50%
   0.030                  2.71%              25.39%     +22.67%
   0.100                  0.00%               3.49%      +3.49%


In [ ]:
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import accuracy_score
from tensorflow.keras.models import load_model
import numpy as np
import cv2
import os

# --- Configuration ---
JPEG_QUALITY = 50
TARGET_SIZE = (240, 240)
BATCH_SIZE = 16

# --- Helper Functions for the Defense ---

def jpeg_compress_opencv(numpy_image_bgr, quality=75):
    """Applies in-memory JPEG compression to a BGR NumPy image array."""
    encode_param = [int(cv2.IMWRITE_JPEG_QUALITY), quality]
    result, encoded_image = cv2.imencode('.jpg', numpy_image_bgr, encode_param)
    if not result:
        raise ValueError("Could not encode image to JPEG")
    return cv2.imdecode(encoded_image, 1)

def jpeg_defense_preprocessor(image_rgb):
    """
    A preprocessing function for ImageDataGenerator that applies the JPEG defense.
    It handles the required RGB <-> BGR color space conversions.
    
    Args:
        image_rgb (np.array): An image in RGB format (from ImageDataGenerator).
    
    Returns:
        np.array: The defended image, also in RGB format.
    """
    # 1. Convert from RGB (Keras default) to BGR (OpenCV default)
    image_bgr = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR)
    
    # 2. Apply the JPEG compression defense on the BGR image
    defended_bgr = jpeg_compress_opencv(image_bgr, quality=JPEG_QUALITY)
    
    # 3. Convert back from BGR to RGB before returning to Keras
    defended_rgb = cv2.cvtColor(defended_bgr, cv2.COLOR_BGR2RGB)
    
    return defended_rgb

# --- Main Evaluation Script ---

# Load the model
# loaded_model_path = '../model.keras'
loaded_model_path = '/home/youssef-abuzeid/Uni/GP/org/Adversarial-Attacks-on-Deeplearning-Models/Attacks/CV/adversarial_images_PGD/../model.keras'
model = load_model(loaded_model_path)

# Load the CSV file
csv_path = '/home/youssef-abuzeid/Uni/GP/org/Adversarial-Attacks-on-Deeplearning-Models/Attacks/CV/Attacks Evaluation on Testset/adversarial_images_PGD_new/adversarial_labels.csv'
df = pd.read_csv(csv_path)

# --- Evaluation Loop ---
results_summary = []
epsilons = [0.007, 0.01, 0.03, 0.1]

for eps in epsilons:
    print(f"\n{'='*60}\nEvaluating for Epsilon = {eps}\n{'='*60}")

    # Filter CSV for the current epsilon
    eps_df = df[df['epsilon'] == eps].copy()
    
    # Define paths
    base_dir = '/home/youssef-abuzeid/Uni/GP/org/Adversarial-Attacks-on-Deeplearning-Models/Attacks/CV/Attacks Evaluation on Testset/adversarial_images_PGD_new'
    image_directory = os.path.join(base_dir, f'eps_{eps}')
    
    # The user's code cleans filenames this way, which is fine, but we need
    # the full path for the generator. The `directory` parameter handles this.
    eps_df['filename'] = eps_df['filename'].str.replace(f'eps_{eps}/', '')

    if eps_df.empty:
        print(f"No data found for epsilon={eps}. Skipping.")
        continue

    # --- 1. Evaluate WITHOUT defense ---
    print("\n--- Evaluating Adversarial Images (No Defense) ---")
    adversarial_datagen = ImageDataGenerator() # No preprocessing function
    
    adversarial_generator = adversarial_datagen.flow_from_dataframe(
        dataframe=eps_df,
        directory=image_directory,
        x_col='filename',
        y_col='label',
        target_size=TARGET_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='raw',
        shuffle=False
    )
    
    # Get predictions and true labels
    adv_predictions = model.predict(adversarial_generator)
    y_pred_adv = tf.argmax(adv_predictions, axis=1).numpy()
    y_true = adversarial_generator.labels
    
    # Calculate and store accuracy
    adv_acc = accuracy_score(y_true, y_pred_adv)
    print(f"Accuracy on Adversarial Images: {adv_acc:.4f}")

    # --- 2. Evaluate WITH JPEG defense ---
    print(f"\n--- Evaluating with JPEG Defense (Quality={JPEG_QUALITY}) ---")
    defended_datagen = ImageDataGenerator(
        preprocessing_function=jpeg_defense_preprocessor
    )
    
    defended_generator = defended_datagen.flow_from_dataframe(
        dataframe=eps_df,
        directory=image_directory,
        x_col='filename',
        y_col='label',
        target_size=TARGET_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='raw',
        shuffle=False
    )
    
    # Get predictions and true labels
    def_predictions = model.predict(defended_generator)
    y_pred_def = tf.argmax(def_predictions, axis=1).numpy()
    y_true_def = defended_generator.labels # Should be identical to y_true
    
    # Calculate and store accuracy
    def_acc = accuracy_score(y_true_def, y_pred_def)
    print(f"Accuracy on Defended Images: {def_acc:.4f}")
       PGD    0.007            95                 36.43%              63.18%     +26.74%

    # Store results for the final report
    results_summary.append({
        "Epsilon": eps,
        "Accuracy (Adversarial)": adv_acc,
        "Accuracy (Defended)": def_acc
    })

# --- Final Report ---
print(f"\n\n{'='*65}\nFINAL ACCURACY REPORT (JPEG DEFENSE, Q={JPEG_QUALITY})\n{'='*65}")
report_df = pd.DataFrame(results_summary)
report_df['Improvement'] = report_df['Accuracy (Defended)'] - report_df['Accuracy (Adversarial)']

# Formatting for clear presentation
report_df['Accuracy (Adversarial)'] = report_df['Accuracy (Adversarial)'].apply(lambda x: f"{x:.2%}")
report_df['Accuracy (Defended)'] = report_df['Accuracy (Defended)'].apply(lambda x: f"{x:.2%}")
report_df['Improvement'] = report_df['Improvement'].apply(lambda x: f"+{x:.2%}" if x >= 0 else f"{x:.2%}")

print(report_df.to_string(index=False))
print("="*65)


Evaluating for Epsilon = 0.007

--- Evaluating Adversarial Images (No Defense) ---
Found 516 validated image filenames.


/home/youssef-abuzeid/.local/lib/python3.10/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()
/home/youssef-abuzeid/.local/lib/python3.10/site-packages/keras/src/models/functional.py:238: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: ['Input_Layer']
Received: inputs=Tensor(shape=(16, 240, 240, 3))
  warnings.warn(msg)


32/33 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step

/home/youssef-abuzeid/.local/lib/python3.10/site-packages/keras/src/models/functional.py:238: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: ['Input_Layer']
Received: inputs=Tensor(shape=(None, 240, 240, 3))
  warnings.warn(msg)


33/33 ━━━━━━━━━━━━━━━━━━━━ 97s 3s/step
Accuracy on Adversarial Images: 0.3643

--- Evaluating with JPEG Defense (Quality=50) ---
Found 516 validated image filenames.


...

33/33 ━━━━━━━━━━━━━━━━━━━━ 23s 706ms/step
Accuracy on Defended Images: 0.6453

Evaluating for Epsilon = 0.03

--- Evaluating Adversarial Images (No Defense) ---
Found 516 validated image filenames.


/home/youssef-abuzeid/.local/lib/python3.10/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


33/33 ━━━━━━━━━━━━━━━━━━━━ 14s 426ms/step
Accuracy on Adversarial Images: 0.0271

--- Evaluating with JPEG Defense (Quality=50) ---
Found 516 validated image filenames.


/home/youssef-abuzeid/.local/lib/python3.10/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


11/33 ━━━━━━━━━━━━━━━━━━━━ 1:25 4s/step

In [ ]:
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import accuracy_score
from tensorflow.keras.models import load_model
import numpy as np
import cv2
import os

# --- Configuration ---
JPEG_QUALITIES = [25,50, 75, 95]  # Test different JPEG qualities
TARGET_SIZE = (240, 240)
BATCH_SIZE = 16


# --- Helper Functions for the Defense ---
def jpeg_compress_opencv(numpy_image_bgr, quality=75):
    encode_param = [int(cv2.IMWRITE_JPEG_QUALITY), quality]
    result, encoded_image = cv2.imencode('.jpg', numpy_image_bgr, encode_param)
    if not result:
        raise ValueError("Could not encode image to JPEG")
    return cv2.imdecode(encoded_image, 1)

def jpeg_defense_preprocessor_factory(quality):
    def jpeg_defense_preprocessor(image_rgb):
        image_bgr = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR)
        defended_bgr = jpeg_compress_opencv(image_bgr, quality=quality)
        defended_rgb = cv2.cvtColor(defended_bgr, cv2.COLOR_BGR2RGB)
        return defended_rgb
    return jpeg_defense_preprocessor

# --- Model ---
model_path = '/home/youssef-abuzeid/Uni/GP/org/Adversarial-Attacks-on-Deeplearning-Models/Attacks/CV/adversarial_images_PGD/../model.keras'
model = load_model(model_path)

# --- Attack Types ---
attack_types = {
    "FGSM": {
        "csv": '/home/youssef-abuzeid/Uni/GP/org/Adversarial-Attacks-on-Deeplearning-Models/Attacks/CV/Attacks Evaluation on Testset/adversarial_images_FGSM_new/adversarial_labels.csv',
        "base_dir": '/home/youssef-abuzeid/Uni/GP/org/Adversarial-Attacks-on-Deeplearning-Models/Attacks/CV/Attacks Evaluation on Testset/adversarial_images_FGSM_new'
    },
    "PGD": {
        "csv": '/home/youssef-abuzeid/Uni/GP/org/Adversarial-Attacks-on-Deeplearning-Models/Attacks/CV/Attacks Evaluation on Testset/adversarial_images_PGD_new/adversarial_labels.csv',
        "base_dir": '/home/youssef-abuzeid/Uni/GP/org/Adversarial-Attacks-on-Deeplearning-Models/Attacks/CV/Attacks Evaluation on Testset/adversarial_images_PGD_new'
    }
}

# --- Main Evaluation Loop ---
for attack_name, paths in attack_types.items():
    df = pd.read_csv(paths["csv"])
    print(f"\n\n{'='*70}\nATTACK TYPE: {attack_name}\n{'='*70}")
    results_summary = []
    epsilons = df['epsilon'].unique()

    for eps in epsilons:
        print(f"\n{'-'*60}\nEvaluating for Epsilon = {eps}\n{'-'*60}")
        eps_df = df[df['epsilon'] == eps].copy()
        image_directory = os.path.join(paths["base_dir"], f'eps_{eps}')
        eps_df['filename'] = eps_df['filename'].str.replace(f'eps_{eps}/', '')

        if eps_df.empty or not os.path.isdir(image_directory):
            print(f"Skipping epsilon={eps}: directory not found or CSV empty.")
            continue

        # --- No Defense ---
        print("\n[+] Evaluating Adversarial (No Defense)")
        adv_gen = ImageDataGenerator().flow_from_dataframe(
            dataframe=eps_df,
            directory=image_directory,
            x_col='filename',
            y_col='label',
            target_size=TARGET_SIZE,
            batch_size=BATCH_SIZE,
            class_mode='raw',
            shuffle=False
        )
        y_pred = tf.argmax(model.predict(adv_gen), axis=1).numpy()
        y_true = adv_gen.labels
        adv_acc = accuracy_score(y_true, y_pred)
        print(f"    Accuracy (No Defense): {adv_acc:.4f}")

        # --- JPEG Defense at Different Qualities ---
        for q in JPEG_QUALITIES:
            print(f"[+] Evaluating JPEG Defense (Q={q})")
            jpeg_def = jpeg_defense_preprocessor_factory(q)
            def_gen = ImageDataGenerator(preprocessing_function=jpeg_def).flow_from_dataframe(
                dataframe=eps_df,
                directory=image_directory,
                x_col='filename',
                y_col='label',
                target_size=TARGET_SIZE,
                batch_size=BATCH_SIZE,
                class_mode='raw',
                shuffle=False
            )
            y_def_pred = tf.argmax(model.predict(def_gen), axis=1).numpy()
            def_acc = accuracy_score(def_gen.labels, y_def_pred)
            print(f"    Accuracy (JPEG Q={q}): {def_acc:.4f}")
            results_summary.append({
                "Attack": attack_name,
                "Epsilon": eps,
                "JPEG Quality": q,
                "Accuracy (Adversarial)": adv_acc,
                "Accuracy (Defended)": def_acc,
                "Improvement": def_acc - adv_acc
            })

    # --- Final Summary Report ---
    summary_df = pd.DataFrame(results_summary)
    summary_df['Accuracy (Adversarial)'] = summary_df['Accuracy (Adversarial)'].apply(lambda x: f"{x:.2%}")
    summary_df['Accuracy (Defended)'] = summary_df['Accuracy (Defended)'].apply(lambda x: f"{x:.2%}")
    summary_df['Improvement'] = summary_df['Improvement'].apply(lambda x: f"+{x:.2%}" if x >= 0 else f"{x:.2%}")

    print(f"\n{'='*65}\nFINAL REPORT FOR {attack_name}\n{'='*65}")
    print(summary_df.to_string(index=False))


2025-06-19 12:31:53.532649: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1750325513.550516   17903 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1750325513.555041   17903 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1750325513.566209   17903 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1750325513.566228   17903 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1750325513.566229   17903 computation_placer.cc:177] computation placer alr



ATTACK TYPE: FGSM

------------------------------------------------------------
Evaluating for Epsilon = 0.007
------------------------------------------------------------

[+] Evaluating Adversarial (No Defense)
Found 516 validated image filenames.


/home/youssef-abuzeid/.local/lib/python3.10/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()
/home/youssef-abuzeid/.local/lib/python3.10/site-packages/keras/src/models/functional.py:238: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: ['Input_Layer']
Received: inputs=Tensor(shape=(16, 240, 240, 3))
  warnings.warn(msg)
I0000 00:00:1750325521.777040   18053 service.cc:152] XLA service 0x766a8c003010 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1750325521.777056   18053 service.cc:160]   StreamExecutor device (0): NVIDIA GeForce GTX 1660 Ti, Compute Capability 7.5
2025-06-19 12:32:01.913718: I tensorflow

 5/33 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step

I0000 00:00:1750325527.877479   18053 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


...

33/33 ━━━━━━━━━━━━━━━━━━━━ 1s 41ms/step
    Accuracy (JPEG Q=50): 0.0640
[+] Evaluating JPEG Defense (Q=75)
Found 516 validated image filenames.
 1/33 ━━━━━━━━━━━━━━━━━━━━ 2s 78ms/step

/home/youssef-abuzeid/.local/lib/python3.10/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


33/33 ━━━━━━━━━━━━━━━━━━━━ 1s 42ms/step
    Accuracy (JPEG Q=75): 0.0349
[+] Evaluating JPEG Defense (Q=95)
Found 516 validated image filenames.
 1/33 ━━━━━━━━━━━━━━━━━━━━ 2s 82ms/step

/home/youssef-abuzeid/.local/lib/python3.10/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


33/33 ━━━━━━━━━━━━━━━━━━━━ 2s 51ms/step
    Accuracy (JPEG Q=95): 0.0078

FINAL REPORT FOR PGD
Attack  Epsilon  JPEG Quality Accuracy (Adversarial) Accuracy (Defended) Improvement
   PGD    0.007            25                 36.43%              79.26%     +42.83%
   PGD    0.007            50                 36.43%              75.58%     +39.15%
   PGD    0.007            75                 36.43%              73.26%     +36.82%
   PGD    0.007            95                 36.43%              63.18%     +26.74%
   PGD    0.010            25                 21.71%              68.80%     +47.09%
   PGD    0.010            50                 21.71%              64.53%     +42.83%
   PGD    0.010            75                 21.71%              62.21%     +40.50%
   PGD    0.010            95                 21.71%              48.84%     +27.13%
   PGD    0.030            25                  2.71%              38.37%     +35.66%
   PGD    0.030            50                  2.71%   

In [3]:
!rm **.png

## Adv Training

In [1]:
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import accuracy_score
from tensorflow.keras.models import load_model
import numpy as np
import cv2
import os

# --- Configuration ---
JPEG_QUALITIES = [25,50, 75, 95]  # Test different JPEG qualities
TARGET_SIZE = (240, 240)
BATCH_SIZE = 16


# --- Helper Functions for the Defense ---
def jpeg_compress_opencv(numpy_image_bgr, quality=75):
    encode_param = [int(cv2.IMWRITE_JPEG_QUALITY), quality]
    result, encoded_image = cv2.imencode('.jpg', numpy_image_bgr, encode_param)
    if not result:
        raise ValueError("Could not encode image to JPEG")
    return cv2.imdecode(encoded_image, 1)

def jpeg_defense_preprocessor_factory(quality):
    def jpeg_defense_preprocessor(image_rgb):
        image_bgr = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR)
        defended_bgr = jpeg_compress_opencv(image_bgr, quality=quality)
        defended_rgb = cv2.cvtColor(defended_bgr, cv2.COLOR_BGR2RGB)
        return defended_rgb
    return jpeg_defense_preprocessor

# --- Model ---
model_path = '/home/youssef-abuzeid/Uni/GP/org/Adversarial-Attacks-on-Deeplearning-Models/Attacks/CV/adversarial_images_PGD/../efficientnet_adv_epoch_21.keras'
model = load_model(model_path)

# --- Attack Types ---
attack_types = {
    "FGSM": {
        "csv": '/home/youssef-abuzeid/Uni/GP/org/Adversarial-Attacks-on-Deeplearning-Models/Attacks/CV/Attacks Evaluation on Testset/adversarial_images_FGSM_new_AT/adversarial_labels.csv',
        "base_dir": '/home/youssef-abuzeid/Uni/GP/org/Adversarial-Attacks-on-Deeplearning-Models/Attacks/CV/Attacks Evaluation on Testset/adversarial_images_FGSM_new_AT'
    },
    "PGD": {
        "csv": '/home/youssef-abuzeid/Uni/GP/org/Adversarial-Attacks-on-Deeplearning-Models/Attacks/CV/Attacks Evaluation on Testset/adversarial_images_PGD_new_AT/adversarial_labels.csv',
        "base_dir": '/home/youssef-abuzeid/Uni/GP/org/Adversarial-Attacks-on-Deeplearning-Models/Attacks/CV/Attacks Evaluation on Testset/adversarial_images_PGD_new_AT'
    }
}

# --- Main Evaluation Loop ---
for attack_name, paths in attack_types.items():
    df = pd.read_csv(paths["csv"])
    print(f"\n\n{'='*70}\nATTACK TYPE: {attack_name}\n{'='*70}")
    results_summary = []
    epsilons = df['epsilon'].unique()

    for eps in epsilons:
        print(f"\n{'-'*60}\nEvaluating for Epsilon = {eps}\n{'-'*60}")
        eps_df = df[df['epsilon'] == eps].copy()
        image_directory = os.path.join(paths["base_dir"], f'eps_{eps}')
        eps_df['filename'] = eps_df['filename'].str.replace(f'eps_{eps}/', '')

        if eps_df.empty or not os.path.isdir(image_directory):
            print(f"Skipping epsilon={eps}: directory not found or CSV empty.")
            continue

        # --- No Defense ---
        print("\n[+] Evaluating Adversarial (No Defense)")
        adv_gen = ImageDataGenerator().flow_from_dataframe(
            dataframe=eps_df,
            directory=image_directory,
            x_col='filename',
            y_col='label',
            target_size=TARGET_SIZE,
            batch_size=BATCH_SIZE,
            class_mode='raw',
            shuffle=False
        )
        y_pred = tf.argmax(model.predict(adv_gen), axis=1).numpy()
        y_true = adv_gen.labels
        adv_acc = accuracy_score(y_true, y_pred)
        print(f"    Accuracy (No Defense): {adv_acc:.4f}")

        # --- JPEG Defense at Different Qualities ---
        for q in JPEG_QUALITIES:
            print(f"[+] Evaluating JPEG Defense (Q={q})")
            jpeg_def = jpeg_defense_preprocessor_factory(q)
            def_gen = ImageDataGenerator(preprocessing_function=jpeg_def).flow_from_dataframe(
                dataframe=eps_df,
                directory=image_directory,
                x_col='filename',
                y_col='label',
                target_size=TARGET_SIZE,
                batch_size=BATCH_SIZE,
                class_mode='raw',
                shuffle=False
            )
            y_def_pred = tf.argmax(model.predict(def_gen), axis=1).numpy()
            def_acc = accuracy_score(def_gen.labels, y_def_pred)
            print(f"    Accuracy (JPEG Q={q}): {def_acc:.4f}")
            results_summary.append({
                "Attack": attack_name,
                "Epsilon": eps,
                "JPEG Quality": q,
                "Accuracy (Adversarial)": adv_acc,
                "Accuracy (Defended)": def_acc,
                "Improvement": def_acc - adv_acc
            })

    # --- Final Summary Report ---
    summary_df = pd.DataFrame(results_summary)
    summary_df['Accuracy (Adversarial)'] = summary_df['Accuracy (Adversarial)'].apply(lambda x: f"{x:.2%}")
    summary_df['Accuracy (Defended)'] = summary_df['Accuracy (Defended)'].apply(lambda x: f"{x:.2%}")
    summary_df['Improvement'] = summary_df['Improvement'].apply(lambda x: f"+{x:.2%}" if x >= 0 else f"{x:.2%}")

    print(f"\n{'='*65}\nFINAL REPORT FOR {attack_name}\n{'='*65}")
    print(summary_df.to_string(index=False))


2025-06-22 08:04:41.867745: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1750568681.885917   56220 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1750568681.890901   56220 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1750568681.907465   56220 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1750568681.907481   56220 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1750568681.907483   56220 computation_placer.cc:177] computation placer alr



ATTACK TYPE: FGSM

------------------------------------------------------------
Evaluating for Epsilon = 0.007
------------------------------------------------------------

[+] Evaluating Adversarial (No Defense)
Found 516 validated image filenames.


/home/youssef-abuzeid/.local/lib/python3.10/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()
/home/youssef-abuzeid/.local/lib/python3.10/site-packages/keras/src/models/functional.py:238: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: ['Input_Layer']
Received: inputs=Tensor(shape=(16, 240, 240, 3))
  warnings.warn(msg)
I0000 00:00:1750568692.474134   56428 service.cc:152] XLA service 0x7b2ed4062e30 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1750568692.474149   56428 service.cc:160]   StreamExecutor device (0): NVIDIA GeForce GTX 1660 Ti, Compute Capability 7.5
2025-06-22 08:04:52.629336: I tensorflow

 5/33 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step

I0000 00:00:1750568699.168933   56428 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


...

33/33 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step
    Accuracy (JPEG Q=50): 0.1686
[+] Evaluating JPEG Defense (Q=75)
Found 516 validated image filenames.
 3/33 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step

/home/youssef-abuzeid/.local/lib/python3.10/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


33/33 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step
    Accuracy (JPEG Q=75): 0.1667
[+] Evaluating JPEG Defense (Q=95)
Found 516 validated image filenames.
 1/33 ━━━━━━━━━━━━━━━━━━━━ 2s 77ms/step

/home/youssef-abuzeid/.local/lib/python3.10/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


33/33 ━━━━━━━━━━━━━━━━━━━━ 1s 33ms/step
    Accuracy (JPEG Q=95): 0.1705

FINAL REPORT FOR PGD
Attack  Epsilon  JPEG Quality Accuracy (Adversarial) Accuracy (Defended) Improvement
   PGD    0.007            25                 68.60%              76.74%      +8.14%
   PGD    0.007            50                 68.60%              76.16%      +7.56%
   PGD    0.007            75                 68.60%              74.81%      +6.20%
   PGD    0.007            95                 68.60%              72.67%      +4.07%
   PGD    0.010            25                 61.43%              68.99%      +7.56%
   PGD    0.010            50                 61.43%              67.25%      +5.81%
   PGD    0.010            75                 61.43%              66.28%      +4.84%
   PGD    0.010            95                 61.43%              65.31%      +3.88%
   PGD    0.030            25                 42.64%              42.05%      -0.58%
   PGD    0.030            50                 42.64%   